# JAGC (ResNet-18) no dataset combinado Acne04 + AcneLevel: fila de seeds em paralelo

Treina, na mesma GPU e ao mesmo tempo (3 processos), mais seeds da receita exata do JAGC no **dataset combinado completo (2863 imagens, as mesmas do artigo)**. Elas entram no ensemble com a receita do artigo, que já está na máquina local.

| Job | O que é |
|---|---|
| `comb45`, `comb46`, `comb47` | Dataset combinado completo, seeds 45, 46 e 47 (somam com as seeds 42-44 que já temos) |

Cada job leva ~4h sozinho (o dobro de imagens do a04-only). Se o tempo apertar, `comb45` e `comb46` já bastam. Todos salvam também as probabilidades com TTA (flip horizontal). Resultados em `Meu Drive/jagc_results/`.

**Antes de começar:** suba `jagc_colab_bundle_v2.zip` para a raiz do Drive e use *GPU A100*.
Se a sessão cair, rode de novo as células 1, 2, 3 e 5: os folds já concluídos são pulados.

In [ ]:
# 1) GPU, Drive e CPUs
!nvidia-smi -L
import os
print("CPUs:", os.cpu_count())
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Usa o torch que o Colab já traz
!pip -q install scikit-learn pillow
import torch, torchvision
print(torch.__version__, torchvision.__version__, "GPU:", torch.cuda.is_available())

In [ ]:
# 3) Descompacta o pacote no disco local
!rm -rf /content/pibic_final
!unzip -q /content/drive/MyDrive/jagc_colab_bundle_v2.zip -d /content
!ls /content/pibic_final/data

In [ ]:
# 4) Teste rápido (~4 min): 10 épocas, fold 0 do combinado completo, com TTA. Confere se tudo funciona antes de lançar a fila.
%cd /content/pibic_final/training
!python run_jagc_original_replica.py --fold 0 --arch resnet18 --epochs 10 --name smoke_colab --tta \
    --splits results/kfold_splits_wholeimage_v2.json --manifest-v2 ../data/final_wholeimage_v2/manifest.json \
    --img-dir ../data/jagc_replica_300_combined --num-workers 3 --output-dir /content/smoke_out 2>&1 | grep -v -i "warn\|F.softmax"

In [ ]:
# 5) LANÇA A FILA EM SEGUNDO PLANO (pode rodar de novo a qualquer momento: pula o que já está pronto)
import os, subprocess
OUT = "/content/drive/MyDrive/jagc_results"
JOBS = ["comb45", "comb46", "comb47"]   # em ordem de prioridade
PARALLEL = 3
WORKERS = max(2, (os.cpu_count() - PARALLEL) // PARALLEL)
os.makedirs("/content/logs", exist_ok=True)
subprocess.Popen(
    f"cd /content/pibic_final/training && python colab_queue.py --out {OUT} --parallel {PARALLEL} --workers {WORKERS} --jobs {' '.join(JOBS)}"
    " > /content/logs/queue.log 2>&1", shell=True, start_new_session=True)
print("Fila lançada. Use a célula 6 para acompanhar.")

In [ ]:
# 6) STATUS (rode quantas vezes quiser)
import glob, os, re, time, numpy as np
print(open("/content/logs/queue.log").read()[-1500:])
print("--- progresso por job (folds prontos no Drive) ---")
for d in sorted(glob.glob("/content/drive/MyDrive/jagc_results/jagc_replica_*")):
    fs = sorted(glob.glob(d + "/fold*.npz"))
    print(f"{os.path.basename(d):55s} {len(fs)}/5 folds")
print("--- tarefas em andamento (última linha de cada log) ---")
for f in sorted(glob.glob("/content/logs/*_fold*.log"), key=os.path.getmtime)[-3:]:
    lines = [l for l in open(f).read().splitlines() if "epoch" in l or "ULTIMA" in l or "TTA" in l]
    print(os.path.basename(f), "->", lines[-1] if lines else "(iniciando)")
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# 7) Métricas agregadas de cada job concluído (5 folds), sem TTA e com TTA
import glob, os, numpy as np
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
def rep(P, Y):
    pr = P.argmax(1)
    return f"Acc={accuracy_score(Y,pr)*100:.2f}%  F1m={f1_score(Y,pr,average='macro')*100:.2f}%  QWK={cohen_kappa_score(Y,pr,weights='quadratic'):.3f}"
for d in sorted(glob.glob("/content/drive/MyDrive/jagc_results/jagc_replica_*")):
    fs = sorted(glob.glob(d + "/fold*.npz"))
    if len(fs) < 5: continue
    z = [np.load(f, allow_pickle=True) for f in fs]
    Y = np.concatenate([a["labels"] for a in z])
    print(os.path.basename(d), "| n =", len(Y))
    print("   sem TTA:", rep(np.concatenate([a["probs"] for a in z]), Y))
    if all("probs_tta" in a for a in z):
        print("   com TTA:", rep(np.concatenate([a["probs_tta"] for a in z]), Y))

Quando terminar (ou quando o tempo acabar), baixe a pasta `jagc_results` do Drive para `training/results/colab/` no projeto e me avise: eu junto com as seeds locais e a receita do artigo, monto o ensemble e aplico a correção de viés com validação cruzada entre folds.